In [10]:
import os
import sys
import time
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.master("local[*]").appName("grande").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
ruta_csv = raiz / "data" / "ventas_grande.csv"

esquema = StructType([
    StructField("id_venta", IntegerType()),
    StructField("cliente", StringType()),
    StructField("producto", StringType()),
    StructField("color", StringType()),
    StructField("vendedor", StringType()),
    StructField("monto", IntegerType()),
    StructField("fecha", StringType()),
])


def limpiar_ventas(df):
    return (
        df.dropna(subset=["cliente"])
        .dropDuplicates(["cliente", "producto", "monto", "fecha"])
        .withColumn("monto", F.col("monto").cast("int"))
        .withColumn("fecha", F.to_date("fecha"))
    )


df = spark.read.csv(ruta_csv.as_uri(), header=True, schema=esquema)
print("Filas leídas:", df.count(), "| Particiones:", df.rdd.getNumPartitions())

Filas leídas: 50750 | Particiones: 2


In [11]:
limpio = limpiar_ventas(df)
print("Antes:", df.count(), "| Después:", limpio.count())

Antes: 50750 | Después: 49000


In [12]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # desactiva el broadcast automático

pequena = limpio.select("producto", "color").distinct().withColumn("id_producto", F.monotonically_increasing_id())

limpio.join(pequena, ["producto", "color"]).explain()
limpio.join(F.broadcast(pequena), ["producto", "color"]).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [producto#373, color#487, id_venta#485, cliente#372, vendedor#489, monto#376, fecha#420, id_producto#464L]
   +- SortMergeJoin [producto#373, color#487], [producto#470, color#493], Inner
      :- Sort [producto#373 ASC NULLS FIRST, color#487 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(producto#373, color#487, 200), ENSURE_REQUIREMENTS, [plan_id=1080]
      :     +- Filter isnotnull(color#487)
      :        +- SortAggregate(key=[cliente#372, producto#373, monto#376, fecha#377], functions=[first(id_venta#371, false), first(color#374, false), first(vendedor#375, false)])
      :           +- Sort [cliente#372 ASC NULLS FIRST, producto#373 ASC NULLS FIRST, monto#376 ASC NULLS FIRST, fecha#377 ASC NULLS FIRST], false, 0
      :              +- Exchange hashpartitioning(cliente#372, producto#373, monto#376, fecha#377, 200), ENSURE_REQUIREMENTS, [plan_id=1063]
      :                 +- SortAggregate(key=

In [5]:
t0 = time.time()
pdf = pd.read_csv(ruta_csv)
pdf = pdf.dropna(subset=["cliente"]).drop_duplicates(subset=["cliente", "producto", "monto", "fecha"])
t_pandas = time.time() - t0
print("pandas:", len(pdf), int(pdf["monto"].sum()), f"{t_pandas:.2f} s")

pandas: 49000 13923787602 0.21 s


In [6]:
t0 = time.time()
r = limpiar_ventas(spark.read.csv(ruta_csv.as_uri(), header=True, schema=esquema)) \
    .agg(F.count("*").alias("filas"), F.sum("monto").alias("total")).collect()[0]
t_spark = time.time() - t0
print("Spark :", r["filas"], r["total"], f"{t_spark:.2f} s")

Spark : 49000 13923787602 2.21 s


In [7]:
mensual = (
    limpio.withColumn("mes", F.date_format("fecha", "yyyy-MM"))
    .groupBy("mes", "vendedor")
    .agg(F.sum("monto").alias("total"), F.count("*").alias("ventas"))
    .orderBy("mes", "vendedor")
)
mensual.show(10)
print("Filas del resumen:", mensual.count())

+-------+--------------+--------+------+
|    mes|      vendedor|   total|ventas|
+-------+--------------+--------+------+
|2025-01|    Ana Torres|86574406|   306|
|2025-01|  Andres Mejia|86720781|   306|
|2025-01|  Camilo Ortiz|84074222|   292|
|2025-01| Julian Vargas|82350247|   298|
|2025-01|  Luis Ramirez|75111670|   265|
|2025-01|  Paula Castro|81928619|   287|
|2025-01|   Sofia Gomez|76875829|   269|
|2025-01|Valentina Rios|72321202|   256|
|2025-02|    Ana Torres|69423524|   240|
|2025-02|  Andres Mejia|73120543|   256|
+-------+--------------+--------+------+
only showing top 10 rows

Filas del resumen: 176


In [8]:
(
    limpio.groupBy("cliente").count()
    .groupBy("count").agg(F.count("*").alias("clientes"))
    .orderBy("count")
    .show(20)
)

+-----+--------+
|count|clientes|
+-----+--------+
|    1|     359|
|    2|     925|
|    3|    1498|
|    4|    1730|
|    5|    1720|
|    6|    1466|
|    7|    1000|
|    8|     593|
|    9|     330|
|   10|     172|
|   11|      83|
|   12|      36|
|   13|      10|
|   14|       3|
|   15|       2|
+-----+--------+



In [9]:
spark.stop()